# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, access, and explore the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and Croissant schema references.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and list its properties using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata information
meta = dataset.metadata
print(f"Dataset: {meta.name}\n\nDescription: {meta.description}\n\nVersion: {meta.version}\nPublished: {meta.datePublished}")

## 2. Data Overview
Review record sets, their available fields (columns), and view their Croissant `@id` values. These identifiers are fundamental for extraction and referencing.

In [ ]:
# List all record sets and their fields by @id
if not hasattr(meta, 'record_sets'):
    print("No record sets found in metadata. Checking for 'recordSet' property instead.")
    record_sets = getattr(meta, 'recordSet', [])
else:
    record_sets = meta.record_sets

rs_ids = []
if record_sets:
    print("Available Record Sets (by @id):\n")
    for rs in record_sets:
        print(f"- {rs['@id']}")
        rs_ids.append(rs['@id'])
        if 'field' in rs and rs['field']:
            print("  Fields (by @id):")
            for f in rs['field']:
                fid = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
                print(f"    - {fid}")
        print("")
else:
    print("No record sets found.") # Shouldn't happen for a valid croissant dataset

## 3. Data Extraction
Load data from one or more record sets using `@id` fields for both the record set and individual fields you wish to analyze.
Below we'll load the data from all discovered record sets.

In [ ]:
# If record sets were found, extract all dataframes
dataframes = {}
loaded_rs_ids = []
if rs_ids:
    for rsid in rs_ids:
        print(f"\nLoading records from RecordSet: {rsid}")
        # Each record yields a dict with keys matching field @id or column name
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            loaded_rs_ids.append(rsid)
            print(f"Loaded {len(df)} records, columns (by field @id):\n{df.columns.tolist()}")
            print(df.head(1))
        else:
            print(f"No records found for {rsid}.")
else:
    print("No record set IDs to extract.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by its `@id`) from one of the loaded record sets for EDA:
- We'll filter for records above a value threshold, normalize, and group.
- Replace the variables below as needed for your analysis (see printed columns above).

In [ ]:
# -- Example selection: Adjust these as needed based on the field @id output earlier --
# For demonstration, we'll programmatically pick the first loaded DataFrame and a numeric field heuristic.

import numpy as np

if loaded_rs_ids:
    rsid = loaded_rs_ids[0]
    df = dataframes[rsid]
    # Try to pick first numeric field (int/float columns)
    numeric_field = None
    for col in df.columns:
        # Try to cast the column to numeric
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
            # Try coercing to numeric
            as_num = pd.to_numeric(df[col], errors='ignore')
            if pd.api.types.is_numeric_dtype(as_num):
                numeric_field = col
                df[col] = as_num
                break
        except Exception:
            continue
    if numeric_field:
        print(f"Using field '{numeric_field}' for numeric analysis from RecordSet '{rsid}'.")
        # Remove NA for safe operation
        df = df[df[numeric_field].notna()]
        # Filter by threshold
        try:
            threshold = np.nanpercentile(df[numeric_field], 50) # Median as demonstration threshold
        except Exception:
            threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} remaining.")
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Try to pick a group field (categorical / object with <30 unique values)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() < 30 and df[col].dtype == object:
                group_field = col
                break

        if group_field:
            print(f"Grouping by '{group_field}'.")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field identified for grouping.")
    else:
        print("No numeric field detected for EDA in the first record set.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if grouping was done, the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if loaded_rs_ids and 'numeric_field' in locals() and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # Boxplot by group
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No suitable numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- Load FAIR^2 dataset metadata and records using `mlcroissant` referencing each element by Croissant `@id`.
- Programmatically explore record sets and their fields.
- Perform basic EDA and visualizations on accessible data tables.

Replace example field selections as appropriate with actual `@id` values and refer to the dataset's documentation ([dataset landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa)) for your specific analysis needs.